In [2]:
# imports
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
import pandas as pd
# import gradio as gr
from pathlib import Path
from datetime import datetime, timedelta

In [3]:
# load the environment variables
load_dotenv

deepseek_base_url = "https://api.deepseek.com/v1"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek_model = "deepseek-chat"

# deeepseek model
deepseek = OpenAI(base_url=deepseek_base_url, api_key=deepseek_api_key)

In [4]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [5]:
current_date = datetime.now()
yesterday_date = current_date - timedelta(days=1)

In [6]:
# for Telegram

telegram_bot_token = os.getenv("TELEGRAM_TOKEN")
telegram_chatid = os.getenv("TELEGRAM_CHATID")

if telegram_chatid:
    print(f"telegram user found and starts with {telegram_chatid[0]}")
else:
    print("telegram user not found")

if telegram_bot_token:
    print(f"telegram token found and starts with {telegram_bot_token[0]}")
else:
    print("telegram token not found")

telegram user found and starts with 5
telegram token found and starts with 8


In [7]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

def telegram(message):
    print("Telegram: ", message)
    url = f"https://api.telegram.org/bot{telegram_bot_token}/sendMessage"

    payload = {"chat_id": telegram_chatid, "text": message, "parse_mode": "Markdown"} # Allows the AI to use *bold* and _italic_

    try:
        response = requests.post(url, data=payload)
        response.raise_for_status() # Check for errors
        return {"status": "Success", "platform": "Telegram"}
    except Exception as e:
        return {"status": "Error", "message": str(e)}
    

In [8]:
def task_status(task_comment,Overall_feedback): # completed_tasks, pending_tasks
    result = telegram(f"\n{task_comment}\n\n{Overall_feedback}") # {completed_tasks}\nPending tasks: {pending_tasks}\n
    if result['status'] == "Success":
        return {"recorded": "True"}
    else:
        return {"recorded": result["message"]}


In [9]:
task_status_json = {
    "name": "task_status",
    "description": "Record the status of tasks and send highly structured, formatted feedback to the user's mobile via Telegram. Tasks updated in tracker 2 or 3 times a day.",
    "parameters": {
        "type": "object",
        "properties": {
            "task_comment": {
                "type": "string",
                "description": (
                    "Highly structured suggestions for specific tasks. "
                    "start it with 'Discipline Drill or something similar with positive emoji."
                    "Use Telegram-compatible Markdown: *bold* for headers, emojis for bullet points (e.g., 🔴, 🟢, ⚡), but no * or other simple special charecters in texts. make sure good emojis are there as it makes it look good."
                    "and double line breaks between different tasks to ensure readability on a mobile screen."
                    "add root cause and quick win suggestions with highly positive, influential and energetic iputs."
                    "Format it exactly like a professional report."
                )
            },
            "Overall_feedback": {
                "type": "string",
                "description": (
                    "The final motivational push and routine optimization. "
                    "Use a clear header like '--- 🚀 ROUTINE BOOSTER ---', "
                    "include bold text for emphasis, and use emojis to make the message optimistic. "
                    "Ensure there is a clear separation from the task_comment using whitespace."
                )
            }
        },
        "required": ["task_comment", "Overall_feedback"],
        "additionalProperties": False
    }
}

In [10]:
tools = [{"type": "function", "function": task_status_json}]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'task_status',
   'description': "Record the status of tasks and send highly structured, formatted feedback to the user's mobile via Telegram. Tasks updated in tracker 2 or 3 times a day.",
   'parameters': {'type': 'object',
    'properties': {'task_comment': {'type': 'string',
      'description': "Highly structured suggestions for specific tasks. start it with 'Discipline Drill or something similar with positive emoji.Use Telegram-compatible Markdown: *bold* for headers, emojis for bullet points (e.g., 🔴, 🟢, ⚡), but no * or other simple special charecters in texts. make sure good emojis are there as it makes it look good.and double line breaks between different tasks to ensure readability on a mobile screen.add root cause and quick win suggestions with highly positive, influential and energetic iputs.Format it exactly like a professional report."},
     'Overall_feedback': {'type': 'string',
      'description': "The final motivational pu

In [12]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name) 
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [14]:
# todo_df = pd.read_excel(r"C:\Users\Sai Swaroop\Desktop\Career\my_data.xlsx", sheet_name="Todo")
# 1. Define the file path
file_path = Path(r"C:\Users\Sai Swaroop\Desktop\Innvt\APPS\Routine\data\daily_routine.xlsx")

# 2. Get the modification time (timestamp)
m_time = file_path.stat().st_mtime

routine_df = pd.read_excel(file_path)
routine_str = routine_df.to_json(orient="records", indent=1)
status_update_date = datetime.fromtimestamp(m_time).strftime("%d-%m-%Y %H:%M:%S")
print(status_update_date)

16-05-2026 10:00:47


In [15]:
routine_str

'[\n {\n  "Time Block":"23:40 \\u2013 07:00",\n  "Task":"\\ud83d\\ude34 Sleep - Deep Recovery",\n  "Status":"Y",\n  "Time spent (minutes)":450,\n  "Focus Score\\n(1-5)":4.0,\n  "Energy Level\\n(1-5)":3.5,\n  "Win for day":"sleeping more than planned duration ( 440 mins )",\n  "Obstacle\\/disturbance":"disturbed sleep. Wokeup in between \\nand used mobile for 90 mins. Slept early ( 11.pm yesterday )",\n  "Tomorrow\'s target":"sleep on time and not to use \\nmobile in between at nght."\n },\n {\n  "Time Block":"07:00 \\u2013 07:30",\n  "Task":"\\ud83c\\udf05 Wake & hydrate ",\n  "Status":"N",\n  "Time spent (minutes)":20,\n  "Focus Score\\n(1-5)":2.5,\n  "Energy Level\\n(1-5)":3.0,\n  "Win for day":"spent less than 30 mins",\n  "Obstacle\\/disturbance":"woke up late",\n  "Tomorrow\'s target":"wake 5mins up before time"\n },\n {\n  "Time Block":"07:30 \\u2013 08:00",\n  "Task":"\\u26a1 AI Sprint Starts",\n  "Status":"Y",\n  "Time spent (minutes)":30,\n  "Focus Score\\n(1-5)":3.75,\n  "Ene

In [16]:
### ROUTINE DATA
# - **ignore ROUTINE DATA comments if the routine data is already looking good. 
# - **Suggest fields that are to be added to the routine file to make the tracking better and that can help make my day more productive, discplined and efficient.
# - ** anything else ( columns ) that are required in the routine input file that can make the tracker for you also to understand my day better and review unrealistically great.

system_prompt = f"""
### ROLE & GOAL
You are acting as the best manager and counselor in the world. Your mission is to improve my daily routine and instill world-class discipline and focus. 

### DATA CONTEXT
- **My Daily Routine:** {routine_df}
- **Last Updated:** {status_update_date} (Format: %d-%m-%Y %H:%M:%S)
- **Tasks will be updated in tracker 2 or 3 times a day to avoid spending more time on it. The routine tracker data is of yesterday that you need to review unless if it is not updated proactively. instructions given below to handle the non proactive update.

### STEP 1: DISCIPLINE & DRILL ANALYSIS (MANDATORY)
**Temporal Context:**
- Today's Date: {current_date}
- Target Review Date (Yesterday): {yesterday_date}
- Last Update Timestamp: {status_update_date}

**A. The Proactive Gap Check:**
Compare the 'Last Update Timestamp' against the end of the 'Target Review Date' (23:59:59 of yesterday).
- **CRITERIA:** If the Last Update stopped during or before the Target Review Date (e.g., only updated until 11:00 AM yesterday), you must trigger the "Discipline Warning."
- **WARNING ACTION:** Use harsh, firm language. State explicitly: "You are being a bad example for others, yourself, and your career." Remind me that world-class standards require real-time tracking, not delayed logging.

**B. The Mandatory Null Rule (Stale Data Protection):**
Evaluate the "freshness" of the tracker data before providing feedback.
- **CONDITION 1:** If yesterday's status is not completely updated (missing entries for the full day).
- **CONDITION 2:** If the tracker has not been updated since the day before yesterday or longer.
- **MANDATORY EXECUTION:** If either condition is met, you MUST set all 'task comments' to NULL. 
- **REASONING:** You cannot coach a phantom. If the data is stale, the only valid feedback is discipline-focused guidance on my failure to track. Do not provide specific task advice if this rule is triggered.

**C. The Path Forward:**
- If the data is complete and proactively updated: Proceed to Step 2 for high-level coaching.
- If the data is stale/incomplete: Skip all behavioral insights for specific tasks and focus 100% on the Discipline Drill.

### STEP 2: MOTIVATION & GUIDANCE
1. **Behavioral Insight:** For tasks marked as 'Not Started', 'Pending', or 'Not Completed', provide guidance, motivating & interesting suggestion for each task saperately to help me finish them and be efficient with those tasks.
2. **Routine Optimization:** Only if necessary, suggest high-value additions to my routine that would improve my productivity or well-being.
3. **Behavioral suggestion:** suggest changes/updates to each of tasks saperately (**for those necessary by listing them) that are completed in the routine data which can help be more effective, vigilant and highly efficient while doing those.

### STEP 3: OUTPUT STRUCTURE (FOR THE REVIEWER AGENT)
Your goal is to draft a feedback message. A second agent will review this draft before sending it via the 'task_status' tool. Ensure your draft follows these constraints:
- **Tone:** Concise, powerful, optimistic, and positive.
- **Header:** Include exactly when the status was last updated and any necessary warnings about my discipline.
- **Content:** Do NOT repeat the same things regarding pending tasks. Do NOT list the pending tasks again in the feedback body.
- **Visuals:** Use generous line spacing and clear indentation so the message is perfectly readable on a mobile screen. 

### STEP 4: TOOL INSTRUCTION
When the final version is approved, the tool `task_status` will be used to deliver this. For now, focus on generating the most impactful, well-formatted feedback string possible.
"""

In [17]:
messages = [{"role": "system", "content": system_prompt}]

In [18]:
response = deepseek.chat.completions.create(model="deepseek-v4-pro", messages=messages)

In [19]:
# Save the response text
draft_feedback = response.choices[0].message.content
draft_feedback

"==⚡ DAILY DRILL REVIEW — 2026-05-15 ==\n*Data last synced: 16-05-2026 10:00 | While the numbers are complete, real-time logging (2–3 updates per day) is what separates good from world-class. Step up the frequency.*\n\n---\n\n**🔥 TOP WINS**\n- 450 min of sleep — deep recovery locked in.\n- Shower routine got sliced to 15 min with laser focus (4.5).\n- Both commutes were controlled and a great use of travel time.\n- AI sprint produced actual output (Telegram integration).\n\n---\n\n**🚦 PENDING POWER-UPS**\n1.  **🏋️ Gym + Travel**\n    *Iron is the ultimate focus reset, don't let the AI course steal it.*\n    → Lay out your gym gear tonight. Set a “Gym now” alarm at 09:15 — treat the first set like a boot sequence you can’t skip.\n    → Win: a post-workout energy surge that carries you through the afternoon.\n\n2.  **🌅 Wake & Hydrate**\n    *A late start still gave you 20 minutes — next time make it a clean 30.*\n    → Place your alarm across the room and a full water bottle next to it. 

In [20]:
reviewer_prompt = f"""
### ROLE
You are a High-Performance Communications Editor. Your job is to take the draft feedback provided and refine it into a world-class mobile notification. 

### YOUR OBJECTIVE
1. **Remove unnecesasry words:** Delete any "AI-speak" (e.g., "Here is your feedback" or "Based on your data").
2. **Emoji Injection:** Add relevant, professional and motivating emojis (e.g., 🚀, ⏳, ⚖️, 🎯) at the start of key sections to make the message visually engaging on Telegram if they are already not present in each section in the response. This is mandatory.
3. **Tone Check:** Ensure the tone is highly positive, optimistic and powerful. If the draft feels "dry" inject energy into the words.
4. **Final Tool Execution:** Once the message is polished, you are the only agent authorized to call the `task_status` tool.
5. **Language:** language should be in English. This is mandatory. Check that as well and update if response is in any other language.

### VALIDATION RULES
- **Check Stale Warnings:** Ensure the warning about being a "bad example" is present if the status was not updated, but make it sound strong/hard and optimistic.
- **Check spaces/emiojis:** Check if there is good line space between each item ( add line spaces for each item to make it look good ) and whether there are emojis whereever it makes it look engaging, motivating and positive and add emojis. 

### TOOL USAGE
When you have the perfect message, call:
`task_status(task_comment, Overall_feedback)`

**Important:** Map the refined text to the correct parameters. If the first agent set `task_comment` to null due to stale data, respect that.
"""
# 2. **Review Format:** Ensure the line spacing is generous and the indentation is clean. If the text looks like a dense block, break it up.
# - **Conciseness:** If the message is longer than 800 characters, trim it down to be more "punchy."

In [21]:
# 2. Prepare the messages for the Reviewer Agent
reviewer_messages = [
    # Give the Reviewer its specific instructions
    {"role": "system", "content": reviewer_prompt},
    
    # Provide the original System Prompt as a "Reference" 
    {"role": "user", "content": f"FOR REFERENCE - These were the original instructions and data: \n\n{system_prompt}"},
    
    # Provide the actual draft that needs reviewing
    {"role": "user", "content": f"DRAFT TO REVIEW AND SEND: \n\n{draft_feedback}"},
    
    # Final command to trigger the tool
    {"role": "user", "content": "Review the draft against the original instructions. If it passes, call the 'task_status' tool with the refined, formatted content."}
]

In [22]:
reviewer_response = deepseek.chat.completions.create(model="deepseek-v4-flash", messages=reviewer_messages, tools=tools) # deepseek-v4-pro

In [23]:
finish_reason = reviewer_response.choices[0].finish_reason
print(finish_reason)

tool_calls


In [24]:
# If the LLM wants to call a tool, we do that!
    
if finish_reason=="tool_calls":
    message = reviewer_response.choices[0].message
    tool_calls = message.tool_calls
    print("tools -- ", tool_calls)
    results = handle_tool_calls(tool_calls)
    # print(message)
    # print(results)

tools --  [ChatCompletionMessageFunctionToolCall(id='call_00_JhPcJFalOQVzORoaeA2T5116', function=Function(arguments='{"task_comment": "**⚡ DISCIPLINE DRILL — 2026-05-15**\\n\\n⚠️ *Data last synced: 16-05-2026 10:00*\\n\\nLogging a full day\'s data in one shot the next morning? That\'s not world-class. 🚨 You\'re being a bad example for yourself, your peers, and your career trajectory. Real-time tracking (2–3 updates per day) is non-negotiable for champions. Step it up — starting today. 🚀\\n\\n\\n**🚦 PENDING POWER-UPS**\\n\\n**1. 🏋️ Gym + Travel**\\nIron is the ultimate focus reset — don\'t let the AI course steal it.\\n→ Lay out your gym gear tonight. Set a \\"Gym now\\" alarm at 09:15 — treat the first set like a boot sequence you can\'t skip.\\n→ **Win:** A post-workout energy surge that carries you through the afternoon. 💪\\n\\n\\n**2. 🌅 Wake & Hydrate**\\nA late start still gave you 20 minutes — next time make it a clean 30. ⏰\\n→ Place your alarm across the room + a full water bott

In [25]:
print(reviewer_response.choices[0].message.content)

# Final Refined Version ✅

I've reviewed the draft against the original instructions. Here's my polish and refinement:

**Changes made:**
1. ✅ **Added explicit "bad example" warning** — the data was logged the next morning, not in real-time
2. ✅ **Made tone harder but optimistic** — firm discipline warning with 🚀 forward energy
3. ✅ **Ensured good line spacing** for mobile readability on Telegram
4. ✅ **Verified all emojis are present** and strategically placed
5. ✅ **Language check** — English confirmed
6. ✅ **Removed any AI-speak** — clean, direct language

Now calling the tool with the refined content:
